# Generation-Time Interventions

Every op in notebooks 01–10 analyses a **single forward pass**. This notebook covers the intervention layer: declarative `Intervention` objects that stay active across *every* decode step of `model.generate(...)` — steer a model while it writes, knock out a component for a whole generation, and watch each token's logit-lens trajectory as it is produced.

> The same objects compose with any single-forward op via `model.intervene(...)`, and with chat models via `model.chat(..., interventions=[...])` (see [`10_chat_models.ipynb`](10_chat_models.ipynb)).

In [ ]:
import interpkit
from interpkit import AblateIntervention, SteerIntervention

model = interpkit.load("gpt2")

## Steering during generation

[`03_steering_vectors.ipynb`](03_steering_vectors.ipynb) steered a single forward pass and compared next-token distributions. Here the steering hook stays registered for the prefill **and every KV-cached decode step**, so the vector shapes the whole continuation.

In [ ]:
vector = model.steer_vector(
    positive=" joy happiness delight",
    negative=" fear dread misery",
    at="transformer.h.6",
)

baseline = model.generate("I feel", max_new_tokens=12)

In [ ]:
steered = model.generate(
    "I feel",
    max_new_tokens=12,
    interventions=[SteerIntervention("transformer.h.6", vector=vector, scale=8.0)],
)

## Watching the model think: `capture="lens"`

`capture="lens"` records each generated token's logit-lens trajectory: at every decode step, each block's hidden state is projected through the validated head pipeline. The table shows, per token, what the final block predicted and the **first block that already predicted the emitted token** — a per-token picture of when the model "made up its mind".

`capture="logits"` records each step's raw final logits instead (`steps[i]["logits"]`).

In [ ]:
result = model.generate("The capital of France is", max_new_tokens=6, capture="lens")

# Everything is also available programmatically:
step0 = result["steps"][0]
print(f"token {step0['token']!r} — lens entries: {len(step0['lens'])} blocks")
print("block 6 top-1:", step0["lens"][6]["top1_token"], f"(p={step0['lens'][6]['top1_prob']:.3f})")

## Positional interventions and the KV cache

`positions` are **absolute and prompt-indexed**: generated token *i* sits at position `prompt_len + i`. During incremental decoding each step presents a length-1 window; interpkit's `GenerationContext` maps absolute positions into it automatically.

Two things to know:

- An intervention pinned to one position modifies that token's block output, which **enters the KV cache** — so it influences every later step (by design). Tokens generated *before* the pinned position are byte-identical to baseline.
- Beam search re-feeds tokens and would break the position mapping, so generation interventions support greedy/sampling only (`num_beams=1`).

In [ ]:
prompt = "The capital of France is"
prompt_len = model._prepare(prompt)["input_ids"].shape[-1]

pinned = model.generate(
    prompt,
    max_new_tokens=6,
    interventions=[
        SteerIntervention(
            "transformer.h.6", vector=vector, scale=20.0,
            positions=(prompt_len + 2,),   # only generated token 2's block output
        )
    ],
)

## Ablating a component for a whole generation

Any intervention type works during generation. Mean-ablating an MLP shows what the model writes *without* that component:

In [ ]:
ablated = model.generate(
    "The capital of France is",
    max_new_tokens=8,
    interventions=[AblateIntervention("transformer.h.10.mlp", method="mean")],
)

## Composing with single-forward ops: `model.intervene(...)`

The context manager applies the same interventions to anything you run inside it — lens, DLA, tracing, activations. Hooks are always removed on exit, even on exceptions.

In [ ]:
with model.intervene(AblateIntervention("transformer.h.10.mlp", method="zero")):
    model.lens("The capital of France is", position=-1)

## CLI equivalents

```bash
interpkit generate gpt2 "I feel" \
    --positive " joy" --negative " fear" --at transformer.h.6 --scale 8
interpkit generate gpt2 "The capital of France is" --capture lens
interpkit generate gpt2 "The capital of France is" --ablate-at transformer.h.10.mlp
```

Next: [`12_circuit_discovery_and_lenses.ipynb`](12_circuit_discovery_and_lenses.ipynb) — gradient-based circuit discovery (AtP/EAP), the tuned lens, and max-activating examples.